# Trabalho Final — Benchmark de Otimização Numérica

**Aluno:** Renyer Montefusco Levy  
**Matrícula:** 2242917

## Objetivo

Resolver e comparar problemas de benchmark de otimização não linear utilizando Julia e solvers de otimização. O trabalho está dividido em duas partes:

1. **AMPL/NLP Benchmark** — 47 problemas em formato AMPL.
2. **COCONUT/GAMS Benchmark** — problemas em formato GAMS com solução de referência disponível.

Os solvers considerados são:

- Ipopt
- MadNLP
- NLopt
- Optim
- Uno

Devido às limitações computacionais do equipamento utilizado, todas as execuções são limitadas a **100 iterações** ou ao parâmetro equivalente disponível em cada solver.


## Critério experimental

As execuções registram:

- fonte do problema;
- formato original;
- nome do problema;
- solver utilizado;
- status retornado;
- valor objetivo obtido;
- valor objetivo de referência, quando disponível;
- gap de otimalidade;
- tempo de execução;
- número de iterações, quando disponível;
- mensagem técnica retornada pela execução.

O gap de otimalidade é calculado por:

$$
\text{gap}=\frac{|f_{obtido}-f_{referência}|}{\max(1,|f_{referência}|)}
$$


## 1. Configuração dos caminhos


In [ ]:
# Esta célula define os caminhos principais do experimento.
# Deve ser executada antes das demais células.

MAX_ITER = 100

function detectar_ampl()
    candidatos = String[]

    caminho_path = Sys.which("ampl")
    if caminho_path !== nothing
        push!(candidatos, String(caminho_path))
    end

    push!(candidatos, "C:/AMPL/ampl.exe")
    push!(candidatos, raw"C:\AMPL\ampl.exe")

    for c in candidatos
        if isfile(c)
            return c
        end
    end

    # Última tentativa: usar o comando pelo PATH do sistema.
    return "ampl"
end

AMPL_EXE = detectar_ampl()
AMPL_SOURCE_DIR = raw"C:\AMPL_NLP_47\source"
AMPL_WORK_DIR = raw"C:\AMPL_NLP_47\execucao"
AMPL_RUN_DIR = joinpath(AMPL_WORK_DIR, "run")
AMPL_LOG_DIR = joinpath(AMPL_WORK_DIR, "logs")
RESULTS_DIR = "results"

mkpath(AMPL_WORK_DIR)
mkpath(AMPL_RUN_DIR)
mkpath(AMPL_LOG_DIR)
mkpath(RESULTS_DIR)

AMPL_DISPONIVEL = isfile(AMPL_EXE) || Sys.which("ampl") !== nothing

println("Limite de iterações: ", MAX_ITER)
println("AMPL: ", AMPL_EXE)
println("Pasta AMPL/NLP: ", AMPL_SOURCE_DIR)
println("Pasta de resultados: ", RESULTS_DIR)
println("AMPL disponível? ", AMPL_DISPONIVEL)
println("Pasta dos modelos AMPL encontrada? ", isdir(AMPL_SOURCE_DIR))


## 2. Pacotes Julia


In [ ]:
import Pkg

# Pacotes externos necessários para a execução principal.
pacotes_externos = [
    "DataFrames",
    "CSV",
    "JuMP",
    "Ipopt"
]

for pacote in pacotes_externos
    try
        @eval using $(Symbol(pacote))
        println("OK: ", pacote)
    catch erro
        println("Instalando pacote: ", pacote)
        Pkg.add(pacote)
        @eval using $(Symbol(pacote))
        println("OK: ", pacote)
    end
end

# Bibliotecas padrão do Julia.
using Dates
using Statistics
using Printf

println("Pacotes principais carregados.")
println("DataFrame definido? ", isdefined(Main, :DataFrame))
println("Model definido? ", isdefined(Main, :Model))


## 3. Estrutura da tabela de resultados


In [ ]:
if !isdefined(Main, :DataFrame)
    error("Execute primeiro a célula 2 — Pacotes Julia. Ela deve terminar com: DataFrame definido? true")
end

resultados = DataFrame(
    parte = String[],
    fonte = String[],
    formato = String[],
    problema = String[],
    solver = String[],
    status = String[],
    objetivo_obtido = Float64[],
    objetivo_referencia = Float64[],
    gap = Float64[],
    tempo_segundos = Float64[],
    iteracoes = Union{Missing, Int}[],
    mensagem = String[]
)

function calcular_gap(objetivo_obtido, objetivo_referencia)
    if isnan(objetivo_obtido) || isnan(objetivo_referencia)
        return NaN
    end
    return abs(objetivo_obtido - objetivo_referencia) / max(1.0, abs(objetivo_referencia))
end

function texto_curto(txt; limite = 300)
    s = replace(string(txt), '\n' => ' ')
    return length(s) <= limite ? s : s[1:limite] * "..."
end

function registrar_resultado!(;
    parte,
    fonte,
    formato,
    problema,
    solver,
    status,
    objetivo_obtido = NaN,
    objetivo_referencia = NaN,
    tempo_segundos = NaN,
    iteracoes = missing,
    mensagem = ""
)
    gap = calcular_gap(objetivo_obtido, objetivo_referencia)

    push!(
        resultados,
        (
            parte,
            fonte,
            formato,
            problema,
            solver,
            string(status),
            objetivo_obtido,
            objetivo_referencia,
            gap,
            tempo_segundos,
            iteracoes,
            texto_curto(mensagem)
        )
    )

    return resultados
end

first(resultados, 0)


# Parte 1 — AMPL/NLP Benchmark

A primeira parte utiliza os 47 problemas do benchmark AMPL/NLP. Os modelos são lidos no formato `.mod` e executados pelo AMPL a partir do Julia. Cada solver é testado quando estiver disponível no ambiente.


## 4. Lista dos 47 problemas AMPL/NLP


In [ ]:
problemas_ampl = [
    "arki0003",
    "arki0009",
    "bearing_400",
    "camshape_6400",
    "clnlbeam",
    "cont5_1_l",
    "cont5_2_1_l",
    "cont5_2_2_l",
    "cont5_2_3_l",
    "cont5_2_4_l",
    "corkscrw",
    "dirichlet120",
    "dtoc1nd",
    "dtoc2",
    "elec_400",
    "ex1_160",
    "ex1_320",
    "ex4_2_160",
    "ex4_2_320",
    "ex8_2_2",
    "ex8_2_3",
    "gasoil_3200",
    "henon120",
    "lane_emden120",
    "marine_1600",
    "NARX_CFy",
    "nql180",
    "optmass",
    "pinene_3200",
    "qcqp500-3c",
    "qcqp500-3nc",
    "qcqp750-2c",
    "qcqp750-2nc",
    "qcqp1000-1nc",
    "qcqp1000-2c",
    "qcqp1000-2nc",
    "qcqp1500-1c",
    "qcqp1500-1nc",
    "qssp180",
    "robot_1600",
    "robot_a",
    "robot_b",
    "robot_c",
    "rocket_12800",
    "steering_12800",
    "svanberg",
    "WM_CFy"
]

arquivos_ampl = DataFrame(
    problema = problemas_ampl,
    arquivo = [joinpath(AMPL_SOURCE_DIR, p * ".mod") for p in problemas_ampl],
    existe = [isfile(joinpath(AMPL_SOURCE_DIR, p * ".mod")) for p in problemas_ampl]
)

display(arquivos_ampl)
println("Total esperado: ", length(problemas_ampl))
println("Arquivos encontrados: ", count(arquivos_ampl.existe))

if count(arquivos_ampl.existe) != length(problemas_ampl)
    println("Atenção: existem arquivos .mod ausentes. Conferir a pasta AMPL_SOURCE_DIR.")
end


## 5. Solvers previstos para a Parte AMPL/NLP

A execução no formato AMPL é realizada por chamada externa ao AMPL a partir do Julia. O notebook tenta os solvers listados e registra falhas quando algum deles não estiver disponível no ambiente.


In [ ]:
solvers_ampl = ["ipopt", "madnlp", "nlopt", "optim", "uno"]

solver_options_ampl = Dict(
    "ipopt"  => ("ipopt_options",  "max_iter=100 print_level=0"),
    "madnlp" => ("madnlp_options", "max_iter=100"),
    "nlopt"  => ("nlopt_options",  "maxeval=100"),
    "optim"  => ("optim_options",  "iterations=100"),
    "uno"    => ("uno_options",    "max_iter=100")
)

DataFrame(
    solver = solvers_ampl,
    limite_iteracoes = fill(MAX_ITER, length(solvers_ampl))
)


## 6. Funções de execução AMPL/NLP


In [ ]:
function ampl_path(path::AbstractString)
    return replace(path, "\\" => "/")
end

function parse_regex_float(txt, pattern)
    m = match(pattern, txt)
    if m === nothing
        return NaN
    end
    try
        return parse(Float64, m.captures[1])
    catch
        return NaN
    end
end

function parse_regex_int(txt, pattern)
    m = match(pattern, txt)
    if m === nothing
        return missing
    end
    try
        return parse(Int, m.captures[1])
    catch
        return missing
    end
end

function parse_regex_string(txt, pattern)
    m = match(pattern, txt)
    if m === nothing
        return ""
    end
    return strip(string(m.captures[1]))
end

function executar_ampl_nlp(problema::String, solver::String)
    mod_file = joinpath(AMPL_SOURCE_DIR, problema * ".mod")
    run_file = joinpath(AMPL_RUN_DIR, problema * "_" * solver * ".run")
    log_file = joinpath(AMPL_LOG_DIR, problema * "_" * solver * ".log")

    if !AMPL_DISPONIVEL
        registrar_resultado!(
            parte = "Parte 1",
            fonte = "AMPL/NLP Benchmark",
            formato = "AMPL",
            problema = problema,
            solver = solver,
            status = "AMPL_NAO_ENCONTRADO",
            mensagem = "Executável AMPL não localizado pelo Julia. Caminho tentado: " * AMPL_EXE
        )
        return
    end

    if !isfile(mod_file)
        registrar_resultado!(
            parte = "Parte 1",
            fonte = "AMPL/NLP Benchmark",
            formato = "AMPL",
            problema = problema,
            solver = solver,
            status = "ARQUIVO_NAO_ENCONTRADO",
            mensagem = mod_file
        )
        return
    end

    opt_name, opt_value = solver_options_ampl[solver]
    mod_ampl = ampl_path(mod_file)

    script = """
reset;
option solver $solver;
option $opt_name \"$opt_value\";
model \"$mod_ampl\";
solve;
printf \"AMPL_STATUS_BEGIN %s AMPL_STATUS_END\\n\", solve_result;
printf \"AMPL_MESSAGE_BEGIN %s AMPL_MESSAGE_END\\n\", solve_message;
printf \"AMPL_OBJECTIVE_BEGIN %.16g AMPL_OBJECTIVE_END\\n\", _obj[1];
quit;
"""

    open(run_file, "w") do io
        write(io, script)
    end

    tempo = @elapsed begin
        try
            open(log_file, "w") do io
                run(pipeline(Cmd([AMPL_EXE, run_file]), stdout = io, stderr = io))
            end
        catch e
            open(log_file, "a") do io
                println(io, "JULIA_ERROR_BEGIN ", sprint(showerror, e), " JULIA_ERROR_END")
            end
        end
    end

    saida = isfile(log_file) ? read(log_file, String) : ""

    status = parse_regex_string(saida, r"AMPL_STATUS_BEGIN\s*(.*?)\s*AMPL_STATUS_END"s)
    mensagem = parse_regex_string(saida, r"AMPL_MESSAGE_BEGIN\s*(.*?)\s*AMPL_MESSAGE_END"s)
    objetivo = parse_regex_float(saida, r"AMPL_OBJECTIVE_BEGIN\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)\s*AMPL_OBJECTIVE_END")
    iteracoes = parse_regex_int(saida, r"Number of Iterations\.+:\s*(\d+)")

    if isempty(status)
        status = occursin("JULIA_ERROR_BEGIN", saida) ? "ERRO_EXECUCAO" : "SEM_STATUS"
    end

    if isempty(mensagem)
        mensagem = texto_curto(saida)
    end

    registrar_resultado!(
        parte = "Parte 1",
        fonte = "AMPL/NLP Benchmark",
        formato = "AMPL",
        problema = problema,
        solver = solver,
        status = status,
        objetivo_obtido = objetivo,
        objetivo_referencia = NaN,
        tempo_segundos = tempo,
        iteracoes = iteracoes,
        mensagem = mensagem
    )
end


## 7. Teste rápido — AMPL/NLP

Esta célula testa um problema antes da execução completa.


In [ ]:
executar_ampl_nlp("qcqp500-3c", "ipopt")
last(resultados, 1)


## 8. Execução completa — 47 problemas AMPL/NLP

A célula seguinte executa os 47 problemas para os solvers listados. A cada execução, os resultados parciais são salvos em arquivo `.csv`.


In [ ]:
# Execução completa da Parte 1.
# São 47 problemas × 5 solvers.

for solver in solvers_ampl
    println("\n=== Solver AMPL/NLP: ", solver, " ===")
    for problema in problemas_ampl
        println("Executando ", problema, " com ", solver)
        executar_ampl_nlp(problema, solver)
        CSV.write(joinpath(RESULTS_DIR, "resultados_parciais_ampl_nlp.csv"), resultados)
    end
end

CSV.write(joinpath(RESULTS_DIR, "resultados_ampl_nlp_47_solvers.csv"), resultados)
display(resultados)


# Parte 2 — COCONUT/GAMS Benchmark

A segunda parte utiliza problemas da biblioteca COCONUT/GAMS com solução de referência. Nesta etapa, os modelos selecionados são representados em Julia/JuMP a partir do formato GAMS e comparados com o valor de referência conhecido.


## 9. Problemas COCONUT/GAMS selecionados


In [ ]:
problemas_coconut = DataFrame(
    numero = [1062, 1063, 1064],
    problema = ["ex4_1_1", "ex4_1_2", "ex4_1_3"],
    fonte = fill("COCONUT/GAMS Benchmark", 3),
    formato = fill("GAMS convertido para JuMP", 3),
    status_referencia = fill(2, 3),
    variaveis = fill(1, 3),
    restricoes = fill(0, 3),
    objetivo_referencia = [-7.4873123649, -663.5000966105, -443.6717047411]
)

display(problemas_coconut)


## 10. Solvers previstos para a Parte COCONUT/GAMS

Os problemas COCONUT/GAMS selecionados são executados em JuMP. Cada solver é utilizado quando sua interface Julia/JuMP estiver disponível no ambiente.


In [ ]:
solvers_jump = ["Ipopt", "MadNLP", "NLopt", "Optim", "Uno"]

function importar_pacote(pkg::Symbol)
    try
        Core.eval(Main, Expr(:import, pkg))
        return true, ""
    catch e
        return false, sprint(showerror, e)
    end
end

function criar_modelo_jump(solver::String)
    if !isdefined(Main, :Model)
        return nothing, false, "JuMP não foi carregado. Execute a célula 2 — Pacotes Julia."
    end

    if solver == "Ipopt"
        ok, err = importar_pacote(:Ipopt)
        if !ok
            return nothing, false, err
        end
        model = Model(Ipopt.Optimizer)
        try set_optimizer_attribute(model, "max_iter", MAX_ITER) catch end
        try set_silent(model) catch end
        return model, true, ""

    elseif solver == "MadNLP"
        ok, err = importar_pacote(:MadNLP)
        if !ok
            return nothing, false, err
        end
        model = Model(MadNLP.Optimizer)
        try set_optimizer_attribute(model, "max_iter", MAX_ITER) catch end
        try set_silent(model) catch end
        return model, true, ""

    elseif solver == "NLopt"
        ok, err = importar_pacote(:NLopt)
        if !ok
            return nothing, false, err
        end
        model = Model(NLopt.Optimizer)
        try set_optimizer_attribute(model, "algorithm", :LD_MMA) catch end
        try set_optimizer_attribute(model, "maxeval", MAX_ITER) catch end
        try set_silent(model) catch end
        return model, true, ""

    elseif solver == "Optim"
        return nothing, false, "Optim.jl não é usado diretamente como solver JuMP geral nesta estrutura. Resultado registrado como não compatível."

    elseif solver == "Uno"
        ok, err = importar_pacote(:Uno)
        if !ok
            return nothing, false, err
        end
        try
            model = Model(Uno.Optimizer)
            try set_optimizer_attribute(model, "max_iter", MAX_ITER) catch end
            try set_silent(model) catch end
            return model, true, ""
        catch e
            return nothing, false, sprint(showerror, e)
        end
    end

    return nothing, false, "Solver não cadastrado."
end

DataFrame(
    solver = solvers_jump,
    limite_iteracoes = fill(MAX_ITER, length(solvers_jump))
)


## 11. Modelos COCONUT/GAMS convertidos para JuMP


In [ ]:
function montar_modelo_coconut!(model, problema::String)
    if problema == "ex4_1_1"
        @variable(model, -2 <= x <= 11, start = -1.2)
        @NLobjective(
            model,
            Min,
            -x - 3.95*x^2 + 7.1*x^3 + 0.4875*x^4 - 2.08*x^5 + x^6 + 0.1
        )
    elseif problema == "ex4_1_2"
        @variable(model, 1 <= x <= 2, start = 1.091)
        @NLobjective(
            model,
            Min,
            -500*x
            + 2.5*x^2
            + 1.666666666*x^3
            + 1.25*x^4
            + x^5
            + 0.8333333*x^6
            + 0.714285714*x^7
            + 0.625*x^8
            + 0.555555555*x^9
            + x^10
            - 43.6363636*x^11
            + 0.41666666*x^12
            + 0.384615384*x^13
            + 0.357142857*x^14
            + 0.3333333*x^15
            + 0.3125*x^16
            + 0.294117647*x^17
            + 0.277777777*x^18
            + 0.263157894*x^19
            + 0.25*x^20
            + 0.238095238*x^21
            + 0.227272727*x^22
            + 0.217391304*x^23
            + 0.208333333*x^24
            + 0.2*x^25
            + 0.192307692*x^26
            + 0.185185185*x^27
            + 0.178571428*x^28
            + 0.344827586*x^29
            + 0.6666666*x^30
            - 15.48387097*x^31
            + 0.15625*x^32
            + 0.1515151*x^33
            + 0.14705882*x^34
            + 0.14285712*x^35
            + 0.138888888*x^36
            + 0.135135135*x^37
            + 0.131578947*x^38
            + 0.128205128*x^39
            + 0.125*x^40
            + 0.121951219*x^41
            + 0.119047619*x^42
            + 0.116279069*x^43
            + 0.113636363*x^44
            + 0.1111111*x^45
            + 0.108695652*x^46
            + 0.106382978*x^47
            + 0.208333333*x^48
            + 0.408163265*x^49
            + 0.8*x^50
        )
    elseif problema == "ex4_1_3"
        @variable(model, 0 <= x <= 10, start = 6.3)
        @NLobjective(
            model,
            Min,
            0.000089248*x
            - 0.0218343*x^2
            + 0.998266*x^3
            - 1.6995*x^4
            + 0.2*x^5
        )
    else
        error("Problema COCONUT/GAMS não cadastrado: " * problema)
    end
    return model
end


## 12. Execução COCONUT/GAMS


In [ ]:
function executar_coconut_jump(problema::String, solver::String, objetivo_referencia::Float64)
    model, ok, msg = criar_modelo_jump(solver)

    if !ok
        registrar_resultado!(
            parte = "Parte 2",
            fonte = "COCONUT/GAMS Benchmark",
            formato = "GAMS convertido para JuMP",
            problema = problema,
            solver = solver,
            status = "SOLVER_INDISPONIVEL",
            objetivo_referencia = objetivo_referencia,
            mensagem = texto_curto(msg)
        )
        return
    end

    try
        montar_modelo_coconut!(model, problema)

        tempo = @elapsed begin
            optimize!(model)
        end

        status = string(termination_status(model))
        objetivo = has_values(model) ? objective_value(model) : NaN

        registrar_resultado!(
            parte = "Parte 2",
            fonte = "COCONUT/GAMS Benchmark",
            formato = "GAMS convertido para JuMP",
            problema = problema,
            solver = solver,
            status = status,
            objetivo_obtido = objetivo,
            objetivo_referencia = objetivo_referencia,
            tempo_segundos = tempo,
            iteracoes = missing,
            mensagem = ""
        )
    catch e
        registrar_resultado!(
            parte = "Parte 2",
            fonte = "COCONUT/GAMS Benchmark",
            formato = "GAMS convertido para JuMP",
            problema = problema,
            solver = solver,
            status = "ERRO_EXECUCAO",
            objetivo_referencia = objetivo_referencia,
            mensagem = texto_curto(sprint(showerror, e))
        )
    end
end

for row in eachrow(problemas_coconut)
    for solver in solvers_jump
        println("Executando ", row.problema, " com ", solver)
        executar_coconut_jump(row.problema, solver, row.objetivo_referencia)
        CSV.write(joinpath(RESULTS_DIR, "resultados_parciais_coconut_gams.csv"), resultados)
    end
end

CSV.write(joinpath(RESULTS_DIR, "resultados_coconut_gams.csv"), resultados)
display(resultados)


# Consolidação dos resultados


## 13. Resultados completos


In [ ]:
CSV.write(joinpath(RESULTS_DIR, "resultados_completos.csv"), resultados)
display(resultados)


## 14. Resumo por solver


In [ ]:
function media_sem_nan(v)
    vals = [x for x in v if !isnan(x)]
    return isempty(vals) ? NaN : mean(vals)
end

function contar_sucessos(statuses)
    return count(s -> begin
        t = lowercase(string(s))
        occursin("solved", t) || occursin("optimal", t) || occursin("locally", t) || occursin("solucao", t) || occursin("solution", t)
    end, statuses)
end

resumo_solver = combine(
    groupby(resultados, [:parte, :solver]),
    :problema => length => :execucoes,
    :status => contar_sucessos => :sucessos,
    :tempo_segundos => media_sem_nan => :tempo_medio,
    :gap => media_sem_nan => :gap_medio
)

display(resumo_solver)
CSV.write(joinpath(RESULTS_DIR, "resumo_por_solver.csv"), resumo_solver)


## 15. Resumo por problema


In [ ]:
resumo_problema = combine(
    groupby(resultados, [:parte, :problema]),
    :solver => length => :tentativas,
    :status => contar_sucessos => :sucessos,
    :objetivo_obtido => media_sem_nan => :objetivo_medio,
    :tempo_segundos => media_sem_nan => :tempo_medio,
    :gap => media_sem_nan => :gap_medio
)

display(resumo_problema)
CSV.write(joinpath(RESULTS_DIR, "resumo_por_problema.csv"), resumo_problema)


## 16. Melhores resultados por problema


In [ ]:
resultados_com_objetivo = filter(row -> !isnan(row.objetivo_obtido), resultados)

if nrow(resultados_com_objetivo) > 0
    melhores = combine(groupby(resultados_com_objetivo, [:parte, :problema])) do sdf
        idx = argmin(sdf.objetivo_obtido)
        return DataFrame(
            solver = sdf.solver[idx],
            status = sdf.status[idx],
            objetivo_obtido = sdf.objetivo_obtido[idx],
            objetivo_referencia = sdf.objetivo_referencia[idx],
            gap = sdf.gap[idx],
            tempo_segundos = sdf.tempo_segundos[idx]
        )
    end
    display(melhores)
    CSV.write(joinpath(RESULTS_DIR, "melhores_resultados_por_problema.csv"), melhores)
else
    println("Nenhum objetivo numérico registrado.")
end


## 17. Arquivos gerados

Os arquivos de resultado são salvos na pasta `results`:

- `resultados_completos.csv`
- `resultados_ampl_nlp_47_solvers.csv`
- `resultados_coconut_gams.csv`
- `resumo_por_solver.csv`
- `resumo_por_problema.csv`
- `melhores_resultados_por_problema.csv`


In [ ]:
readdir(RESULTS_DIR)
